In [ ]:
import pandas as pd
!pip install kagglehub

In [ ]:
import kagglehub
#this uploads it from kaggle directly
path = kagglehub.dataset_download("nadyinky/sephora-products-and-skincare-reviews")

print("Path to dataset files:", path)

100%|██████████| 147M/147M [00:03<00:00, 47.2MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nadyinky/sephora-products-and-skincare-reviews/versions/2


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import os
import re
from textblob import TextBlob
from collections import Counter

In [ ]:
import os
files = os.listdir(path)
print(files)

['product_info.csv', 'reviews_0-250.csv', 'reviews_750-1250.csv', 'reviews_250-500.csv', 'reviews_1250-end.csv', 'reviews_500-750.csv']


In [ ]:
import pandas as pd
import os

# Load product data
products = pd.read_csv(os.path.join(path, "product_info.csv"))

# Your review files (based on what you showed)
review_files = [
    "reviews_0-250.csv",
    "reviews_250-500.csv",
    "reviews_500-750.csv",
    "reviews_750-1250.csv",
    "reviews_1250-end.csv"
]

# Load and combine
reviews_list = []
for file in review_files:
    temp = pd.read_csv(os.path.join(path, file))
    reviews_list.append(temp)

reviews = pd.concat(reviews_list, ignore_index=True)

print("Reviews shape:", reviews.shape)

/tmp/ipykernel_205/2569142341.py:19: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(os.path.join(path, file))
/tmp/ipykernel_205/2569142341.py:19: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(os.path.join(path, file))
/tmp/ipykernel_205/2569142341.py:19: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  temp = pd.read_csv(os.path.join(path, file))


Reviews shape: (1094411, 19)


In [ ]:
df = reviews.merge(products, on="product_id", how="left")

df.head()

,Unnamed: 0,author_id,rating_x,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,1741593524,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,...,1,0,0,['Clean at Sephora'],Skincare,Cleansers,NaN,0,NaN,NaN
1,1,31423088263,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
2,2,5061282401,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
3,3,6083038851,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0
4,4,47056667835,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",...,0,0,1,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3,24.0,24.0


In [ ]:
df.columns.tolist()
print(df.shape)
print(df[["product_id"]].head())
print(df["product_id"].isna().sum())
df[["rating_x", "rating_y"]].head()
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

(1094411, 45)
  product_id
0    P504322
1    P420652
2    P420652
3    P420652
4    P420652
0


In [ ]:
df = df.sample(50000, random_state=42)

In [ ]:
df[["review_text"]].head()

,review_text
1049894,I have used 2 bottles of this product and it d...
619554,"I love this product, I feel like it really cal..."
634782,Lately I’ve been loving my skin and this toner...
706815,This toner is one of the best purchases I have...
731656,I have been loving all the products so far the...


In [ ]:
df = df.rename(columns={
    "rating_x": "review_rating",
    "rating_y": "product_rating"
})

In [ ]:
df = df.drop(columns=[
    "product_name_y",
    "brand_name_y"
])

In [ ]:
df[["product_id", "review_text", "review_rating", "product_rating", "skin_type", "ingredients"]].head()

,product_id,review_text,review_rating,product_rating,skin_type,ingredients
1049894,P470240,I have used 2 bottles of this product and it d...,1,4.1875,dry,"['Aqua/Water/Eau, Glycerin, Oleth-20, Sodium A..."
619554,P423259,"I love this product, I feel like it really cal...",5,4.2657,combination,"['Water/Eau, Dipropylene Glycol, Glycerin, But..."
634782,P449175,Lately I’ve been loving my skin and this toner...,5,4.3610,oily,"['Water, Glycerin, Alcohol Denat., Dipropylene..."
706815,P4015,This toner is one of the best purchases I have...,5,4.3180,NaN,"['Water, Witch Hazel Water, Butylene Glycol, A..."
731656,P464233,I have been loving all the products so far the...,5,4.2407,combination,"['Water, Octyldodecanol, C12-15 Alkyl Benzoate..."


In [ ]:
!pip install transformers torch -q

In [ ]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_review"] = df["review_text"].apply(clean_text)

In [ ]:
df[["review_text", "clean_review"]].head()

,review_text,clean_review
1049894,I have used 2 bottles of this product and it d...,i have used bottles of this product and it did...
619554,"I love this product, I feel like it really cal...",i love this product i feel like it really calm...
634782,Lately I’ve been loving my skin and this toner...,lately i ve been loving my skin and this toner...
706815,This toner is one of the best purchases I have...,this toner is one of the best purchases i have...
731656,I have been loving all the products so far the...,i have been loving all the products so far the...


In [ ]:
sample_reviews = df["clean_review"].head(5).tolist()

for review in sample_reviews:
    result = sentiment_model(review[:512])[0]
    print(review[:120])
    print(result)
    print("-" * 50)

i have used bottles of this product and it did absolutely nothing it was waste of money for me
{'label': 'NEGATIVE', 'score': 0.6456491351127625}
--------------------------------------------------
i love this product i feel like it really calms down my redness and my skin has felt much softer and has a better textur
{'label': 'POSITIVE', 'score': 0.9983035326004028}
--------------------------------------------------
lately i ve been loving my skin and this toner came at the right time it really keeps my skin hydrated and one tip i hav
{'label': 'POSITIVE', 'score': 0.9983537197113037}
--------------------------------------------------
this toner is one of the best purchases i have ever made i have oily acne prone skin and this toner has helped with keep
{'label': 'POSITIVE', 'score': 0.9712426066398621}
--------------------------------------------------
i have been loving all the products so far they make my face feel amazinggggg i will definitely be buying more no mater 
{'label': 'PO

In [ ]:
def get_hf_sentiment(text):
    text = str(text)[:512]   # keep text short enough for the model
    result = sentiment_model(text)[0]

    if result["label"] == "POSITIVE":
        return result["score"]
    else:
        return -result["score"]

In [ ]:
df_model = df.head(5000).copy()
df_model["sentiment"] = df_model["clean_review"].apply(get_hf_sentiment)

In [ ]:
df_model[["review_text", "sentiment"]].head(10)

,review_text,sentiment
1049894,I have used 2 bottles of this product and it d...,-0.645649
619554,"I love this product, I feel like it really cal...",0.998304
634782,Lately I’ve been loving my skin and this toner...,0.998354
706815,This toner is one of the best purchases I have...,0.971243
731656,I have been loving all the products so far the...,0.999868
37430,I’ve searched for many years online for creams...,0.795922
845174,I love this mask! So soothing! Makes my skin s...,0.999635
570505,"I when Influencer sent me this product, and it...",0.997670
347448,I love how the salve now comes in a convenient...,-0.996519
672831,Really enjoy this product! Has a nice subtle c...,0.999795


In [ ]:
def sentiment_label(score):
    if score > 0.1:
        return "positive"
    elif score < -0.1:
        return "negative"
    else:
        return "neutral"

df_model["sentiment_label"] = df_model["sentiment"].apply(sentiment_label)

In [ ]:
df_model[["review_text", "sentiment", "sentiment_label"]].head(10)

,review_text,sentiment,sentiment_label
1049894,I have used 2 bottles of this product and it d...,-0.645649,negative
619554,"I love this product, I feel like it really cal...",0.998304,positive
634782,Lately I’ve been loving my skin and this toner...,0.998354,positive
706815,This toner is one of the best purchases I have...,0.971243,positive
731656,I have been loving all the products so far the...,0.999868,positive
37430,I’ve searched for many years online for creams...,0.795922,positive
845174,I love this mask! So soothing! Makes my skin s...,0.999635,positive
570505,"I when Influencer sent me this product, and it...",0.997670,positive
347448,I love how the salve now comes in a convenient...,-0.996519,negative
672831,Really enjoy this product! Has a nice subtle c...,0.999795,positive


In [ ]:
product_sentiment = df_model.groupby("product_id").agg(
    avg_sentiment=("sentiment", "mean"),
    review_count=("sentiment", "count"),
    avg_review_rating=("review_rating", "mean")
).reset_index()

In [ ]:
product_sentiment.head()

,product_id,avg_sentiment,review_count,avg_review_rating
0,P107306,0.964631,1,4.000000
1,P114902,-0.111970,9,4.444444
2,P12045,0.109571,15,4.333333
3,P122651,0.999622,1,5.000000
4,P122661,0.997536,1,5.000000


In [ ]:
product_info_simple = df_model[["product_id", "product_name_x", "brand_name_x"]].drop_duplicates()

product_sentiment = product_sentiment.merge(
    product_info_simple,
    on="product_id",
    how="left"
)

product_sentiment = product_sentiment.rename(columns={
    "product_name_x": "product_name",
    "brand_name_x": "brand_name"
})

In [ ]:
product_sentiment.head()

,product_id,avg_sentiment,review_count,avg_review_rating,product_name,brand_name
0,P107306,0.964631,1,4.000000,Renewing Eye Cream,Murad
1,P114902,-0.111970,9,4.444444,Goodbye Acne Max Complexion Correction Pads,Peter Thomas Roth
2,P12045,0.109571,15,4.333333,Grape Water Moisturizing Face Mist,Caudalie
3,P122651,0.999622,1,5.000000,Clarifying Lotion 1,CLINIQUE
4,P122661,0.997536,1,5.000000,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE


In [ ]:
def calc_polarization(group):
    positive_count = (group["sentiment"] > 0.1).sum()
    negative_count = (group["sentiment"] < -0.1).sum()
    return positive_count / (negative_count + 1)

polarization = df_model.groupby("product_id").apply(calc_polarization).reset_index(name="polarization_score")

/tmp/ipykernel_205/136980524.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  polarization = df_model.groupby("product_id").apply(calc_polarization).reset_index(name="polarization_score")


In [ ]:
product_sentiment = product_sentiment.merge(
    polarization,
    on="product_id",
    how="left"
)

In [ ]:
product_sentiment.head()

,product_id,avg_sentiment,review_count,avg_review_rating,product_name,brand_name,polarization_score
0,P107306,0.964631,1,4.000000,Renewing Eye Cream,Murad,1.000000
1,P114902,-0.111970,9,4.444444,Goodbye Acne Max Complexion Correction Pads,Peter Thomas Roth,0.666667
2,P12045,0.109571,15,4.333333,Grape Water Moisturizing Face Mist,Caudalie,1.000000
3,P122651,0.999622,1,5.000000,Clarifying Lotion 1,CLINIQUE,1.000000
4,P122661,0.997536,1,5.000000,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,1.000000


In [ ]:
skin_type_sentiment = df_model.groupby(["product_id", "skin_type"]).agg(
    avg_sentiment=("sentiment", "mean"),
    review_count=("sentiment", "count")
).reset_index()

In [ ]:
skin_type_sentiment.head(10)

,product_id,skin_type,avg_sentiment,review_count
0,P107306,combination,0.964631,1
1,P114902,combination,-0.595876,5
2,P12045,combination,0.072246,6
3,P12045,dry,-0.024602,4
4,P12045,normal,0.999765,1
5,P122651,dry,0.999622,1
6,P122718,combination,-0.944720,2
7,P122718,normal,0.999259,1
8,P122762,combination,0.999881,1
9,P122762,dry,-0.999689,1


In [ ]:
def make_summary(row):
    if row["avg_sentiment"] > 0.2:
        tone = "mostly positive"
    elif row["avg_sentiment"] < -0.1:
        tone = "mostly negative"
    else:
        tone = "mixed"

    return f"Reviews for this product are {tone}, with an average sentiment score of {row['avg_sentiment']:.2f}."

product_sentiment["summary_text"] = product_sentiment.apply(make_summary, axis=1)

In [ ]:
product_sentiment[["product_id", "product_name", "avg_sentiment", "summary_text"]].head()

,product_id,product_name,avg_sentiment,summary_text
0,P107306,Renewing Eye Cream,0.964631,"Reviews for this product are mostly positive, ..."
1,P114902,Goodbye Acne Max Complexion Correction Pads,-0.111970,"Reviews for this product are mostly negative, ..."
2,P12045,Grape Water Moisturizing Face Mist,0.109571,"Reviews for this product are mixed, with an av..."
3,P122651,Clarifying Lotion 1,0.999622,"Reviews for this product are mostly positive, ..."
4,P122661,7 Day Face Scrub Cream Rinse-Off Formula,0.997536,"Reviews for this product are mostly positive, ..."


In [ ]:
ingredient_list = [
    "niacinamide",
    "retinol",
    "vitamin c",
    "hyaluronic acid",
    "salicylic acid",
    "ceramides",
    "peptides",
    "glycerin"
]

In [ ]:
def find_ingredients(text):
    found = []
    for ingredient in ingredient_list:
        if ingredient in text:
            found.append(ingredient)
    return found

df_model["ingredients_mentioned"] = df_model["clean_review"].apply(find_ingredients)

In [ ]:
df_model[["review_text", "ingredients_mentioned"]].head(10)

,review_text,ingredients_mentioned
1049894,I have used 2 bottles of this product and it d...,[]
619554,"I love this product, I feel like it really cal...",[]
634782,Lately I’ve been loving my skin and this toner...,[]
706815,This toner is one of the best purchases I have...,[]
731656,I have been loving all the products so far the...,[]
37430,I’ve searched for many years online for creams...,[]
845174,I love this mask! So soothing! Makes my skin s...,[]
570505,"I when Influencer sent me this product, and it...",[]
347448,I love how the salve now comes in a convenient...,[]
672831,Really enjoy this product! Has a nice subtle c...,[]


In [ ]:
positive_reviews = df_model[df_model["sentiment"] > 0.1].copy()
ingredient_counts = (
    positive_reviews.explode("ingredients_mentioned")
    .dropna(subset=["ingredients_mentioned"])
    .groupby(["product_id", "ingredients_mentioned"])
    .size()
    .reset_index(name="count")
)

In [ ]:
top_ingredients = (
    ingredient_counts.sort_values(["product_id", "count"], ascending=[True, False])
    .groupby("product_id")
    .head(3)
)
top_ingredients.head(10)

,product_id,ingredients_mentioned,count
0,P12045,hyaluronic acid,1
1,P173622,retinol,1
2,P232327,retinol,1
3,P309306,retinol,1
4,P375854,niacinamide,1
5,P392245,retinol,1
6,P392246,hyaluronic acid,1
7,P392246,niacinamide,1
8,P394397,vitamin c,1
9,P400259,vitamin c,2


In [ ]:
final_product_output = product_sentiment[[
    "product_id",
    "product_name",
    "brand_name",
    "avg_sentiment",
    "review_count",
    "avg_review_rating",
    "polarization_score",
    "summary_text"
]].copy()
final_product_output.head()

,product_id,product_name,brand_name,avg_sentiment,review_count,avg_review_rating,polarization_score,summary_text
0,P107306,Renewing Eye Cream,Murad,0.964631,1,4.000000,1.000000,"Reviews for this product are mostly positive, ..."
1,P114902,Goodbye Acne Max Complexion Correction Pads,Peter Thomas Roth,-0.111970,9,4.444444,0.666667,"Reviews for this product are mostly negative, ..."
2,P12045,Grape Water Moisturizing Face Mist,Caudalie,0.109571,15,4.333333,1.000000,"Reviews for this product are mixed, with an av..."
3,P122651,Clarifying Lotion 1,CLINIQUE,0.999622,1,5.000000,1.000000,"Reviews for this product are mostly positive, ..."
4,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,0.997536,1,5.000000,1.000000,"Reviews for this product are mostly positive, ..."


In [ ]:
final_review_output = df_model[[
    "product_id",
    "review_text",
    "review_rating",
    "skin_type",
    "sentiment",
    "sentiment_label"
]].copy()

In [ ]:
final_review_output.head()

,product_id,review_text,review_rating,skin_type,sentiment,sentiment_label
1049894,P470240,I have used 2 bottles of this product and it d...,1,dry,-0.645649,negative
619554,P423259,"I love this product, I feel like it really cal...",5,combination,0.998304,positive
634782,P449175,Lately I’ve been loving my skin and this toner...,5,oily,0.998354,positive
706815,P4015,This toner is one of the best purchases I have...,5,NaN,0.971243,positive
731656,P464233,I have been loving all the products so far the...,5,combination,0.999868,positive


In [ ]:
top_ingredients_grouped = (
    top_ingredients.groupby("product_id")["ingredients_mentioned"]
    .apply(list)
    .reset_index(name="top_praised_ingredients")
)

top_ingredients_grouped.head()

,product_id,top_praised_ingredients
0,P12045,[hyaluronic acid]
1,P173622,[retinol]
2,P232327,[retinol]
3,P309306,[retinol]
4,P375854,[niacinamide]


In [ ]:
kristen_output = final_product_output.merge(
    top_ingredients_grouped,
    on="product_id",
    how="left"
)

kristen_output.head()

,product_id,product_name,brand_name,avg_sentiment,review_count,avg_review_rating,polarization_score,summary_text,top_praised_ingredients
0,P107306,Renewing Eye Cream,Murad,0.964631,1,4.000000,1.000000,"Reviews for this product are mostly positive, ...",NaN
1,P114902,Goodbye Acne Max Complexion Correction Pads,Peter Thomas Roth,-0.111970,9,4.444444,0.666667,"Reviews for this product are mostly negative, ...",NaN
2,P12045,Grape Water Moisturizing Face Mist,Caudalie,0.109571,15,4.333333,1.000000,"Reviews for this product are mixed, with an av...",[hyaluronic acid]
3,P122651,Clarifying Lotion 1,CLINIQUE,0.999622,1,5.000000,1.000000,"Reviews for this product are mostly positive, ...",NaN
4,P122661,7 Day Face Scrub Cream Rinse-Off Formula,CLINIQUE,0.997536,1,5.000000,1.000000,"Reviews for this product are mostly positive, ...",NaN


In [ ]:
kristen_output = kristen_output[[
    "product_id",
    "avg_sentiment",
    "polarization_score",
    "summary_text",
    "top_praised_ingredients"
]]

In [ ]:
kristen_output.to_csv("kristen_nlp_output.csv", index=False)
skin_type_sentiment.to_csv("skin_type_sentiment.csv", index=False)
final_review_output.to_csv("final_review_output.csv", index=False)

Polarization scores:
- a higher value means more positive than negative reviews
- a lower value means more balanced or more negative feedback

In [ ]:
kristen_output.head(20)

,product_id,avg_sentiment,polarization_score,summary_text,top_praised_ingredients
0,P107306,0.964631,1.000000,"Reviews for this product are mostly positive, ...",NaN
1,P114902,-0.111970,0.666667,"Reviews for this product are mostly negative, ...",NaN
2,P12045,0.109571,1.000000,"Reviews for this product are mixed, with an av...",[hyaluronic acid]
3,P122651,0.999622,1.000000,"Reviews for this product are mostly positive, ...",NaN
4,P122661,0.997536,1.000000,"Reviews for this product are mostly positive, ...",NaN
5,P122718,0.221704,1.000000,"Reviews for this product are mostly positive, ...",NaN
6,P122762,0.333118,1.000000,"Reviews for this product are mostly positive, ...",NaN
7,P122774,0.001437,0.666667,"Reviews for this product are mixed, with an av...",NaN
8,P122782,0.001616,0.500000,"Reviews for this product are mixed, with an av...",NaN
9,P122876,0.865001,1.000000,"Reviews for this product are mostly positive, ...",NaN


In [1]:
!git clone https://github.com/emilynnguyen1/Skintelligent.git

Cloning into 'Skintelligent'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 15 (delta 3), reused 15 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), done.
Resolving deltas: 100% (3/3), done.
